# Inference
Inference put simply is when we actually use the model. Given that we are done training and have our updated weights, we now can feed new/unseen tokens into the model and generate predictions. Inference occurs when you open up claude.ai and ask it to explain how attention works.

The biggest difference between inference and training is that now we have no use for targets, so not only do we not provide targets, but we also do not evaluate the model on correctness. Similarly, we need not worry about masking as future tokens do not exist yet nor gradients as we are not updating weights.
```
input_1: the dog chased
output_1: the dog chased the
_______________________
input_2: the dog chased the
output_2: the dog chased the cat
_______________________
input_3: the dog chased the cat
output_3: the dog chased the cat through
_______________________
input_4: the dog chased the cat through
output_4: the dog chased the cat through the
_______________________
input_5: the dog chased the cat through the
output_5: the dog chased the cat through the yard
_______________________
```

## Sampling
The natural next question is: how does the model select the next token? We already discussed top_k sampling (which limits the last dimension of logits from ```vocab_size``` to `k`). And in our training code, we see that our sampling code is simply:
```
probs = F.softmax(last, dim = -1)
next_token = torch.multinomial(probs, num_samples = 1)
```
where the function we use remains stochastic/nondeterministic. For instance, if "the" has a 95% chance of being the next token, then ```torch.multinomial``` will select "the" with a 95% chance.

### Why not take argmax?
Randomness makes the output more natural and varied. But we certainly can take argmax, which makes the process deterministic. We simply do so by setting temperature = 0 so that the word with the highest probability is always going to be the next one sampled. But this can be very boring and ineffective, since given the same input, the model will always predict the same next sequence, which is not representative of true language.

### Temperature Refresher
Recall:

$σ(z)_i = \frac{e^{\frac{z_i}{T}}}{∑e^{\frac{z_j}{T}}}$


When temp > 1, our distribution is more spread out / random.

When temp = 1, softmax is unchanged.

When temp < 1, our distribution is more peaky / higher logits amplified.

### Top K refresher
Selecting one token out of vocab_size (50257) can lead to some bad selections, so we limit the model to choosing from the k most likely tokens. We just have to make sure we re-normalize to ensure the probabilities sum to 1 (which is handled automatically by PyTorch).

### Top P (nuclus sampling)
Top P sampling works similarly to Top K sampling with a bit of a twist: rather than sampling from the top K tokens, we sample from the smallest set of tokens whose cumulative probabilities exceed P. We do this by sorting the tokens by probability.

```
tokens_probs = [.32, .27 .23, .10, ...]

if P = .50:
top_p = [.32, .27]

if P = .80:
top_p = [.32, .27, .23]

if P = .90
top_p = [.32, .27 .23, .10]
```
The main draw towards Top P sampling over Top K sampling is that Top P adapts to the model's confidence. When the model is very confident, we do not waste resourcing selecting K tokens with low probabilities. When the model is not confident, we select an appropriate amount of tokens as opposed to accidentally excluding some with real probability.

### Beam Search
What if we did not just have to pick one token? Sampling can lead us down a locally good path but a globally bad path. For example, "The cat" might be the best first two tokens, but "The dog ate" might be a better full sentence than "The cat slept." Greedy would commit to "The cat" and never consider "The dog."
Beam search keeps multiple candidates alive simultaneously, so it can find better full sequences even if the first token was not the absolute best. Beam search ultimately is deterministic, so it is often used for translation as opposed to generating text. It is also very slow. Using [Raiyan](https://github.com/raiyanyahya/how-to-train-your-gpt/blob/master/chapters/09_inference.md)'s example:

```
Beam width = 3:

Step 1: "The" → 3 best next tokens: ["cat"(0.3), "dog"(0.2), "man"(0.1)]
Step 2: "The cat" → 3 best continuations: ["sat"(0.4), "is"(0.2), "was"(0.15)]
        "The dog" → 3 best: ["ran"(0.35), "is"(0.2), "barked"(0.1)]
        "The man" → 3 best: ["walked"(0.3), "said"(0.25), "is"(0.1)]
        Pick top 3 overall sequences:
        "The cat sat" (0.3×0.4=0.12), "The dog ran" (0.2×0.35=0.07), ...
```
### Repetition Penalty
This is relatively straightforward. We just penalize tokens that have appeared recently so that the model does not repeat itself "I I I I"
```
for each candidate token:
  penalty = 1.0 if not in recent history
  penalthy < 1.0 if in recent history (by varying degree)
logits *= penalty
```


---


## KV Caching
The solution for managing the context far more effectively is KV Caching.
A simple example (pretend we generate word by word):
```
input_1: the dog chased
output_1: the dog chased the
_______________________
input_2: the dog chased the
output_2: the dog chased the cat
_______________________
input_3: the dog chased the cat
output_3: the dog chased the cat through
_______________________
input_4: the dog chased the cat through
output_4: the dog chased the cat through the
_______________________
input_5: the dog chased the cat through the
output_5: the dog chased the cat through the yard
_______________________

pseudocode:
for _ in range(max_new_tokens):
  logits, _ = model(input_ids)
  next_token = sample(logits[:, -1, :])
  input_ids = torch.cat([input_ids, next_token], dim=1)
```
Let's briefly revisit what we know about how a token is processed in the attention step. ```input_ids``` come in: for each token we compute a Q,K,V vector. However, with autoregressive generation (where we include the generated text as well), think about how many times we compute the K,V vectors for the first token (and the second token, and third...). In essence, we recompute the K,V matrices for every token in every step. KV Cache eliminates a lot of this redundancy by storing K, V vectors so we only compute them for new tokens.
```
Step 1: Process "The"           → Store K["The"], V["The"] in cache
Step 2: Process "dog"           → Reuse K,V for "The", compute K,V for "dog"
Step 3: Process "chased"        → Reuse K,V for "The","dog", compute for "chased"
...
Step 100: Process "yard"        → Reuse K,V for 99 tokens, compute 1 new
```
Each generation step adds one new row to K and V (recall the vectors are of shape ```seq_len * dim_per_head```). After 100 steps, your KV cache has 100 rows per layer per head. You only compute the new row for the new token, then append it.

The memory cost of KV Cache can be represented as ```2* num_layers * head_count * seq_len * dim_per_head```which can be a lot of memory with large models.


In [3]:
import torch
def load_checkpoint(checkpoint_path: str, device: torch.device):
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model = GPT(checkpoint["config"])
    model.load_state_dict(checkpoint["model_state_dict"])

    model = model.to(device)  # Move to GPU
    model.eval()              # Disable dropout for inference
    print(f"Loaded model from step {checkpoint['step']}, "
          f"loss: {checkpoint['loss']:.4f}")
    return model